# Steam market analysis for Ubisoft: 02 Genres
Representation, player reception, publishers' favourite genres and the most lucrative genres. A game can have several genres: counts are made on `genres_df` (one row per game and genre).

In [0]:
%run ./setup_data

# Steam analysis: shared data setup
Loads and cleans the Steam catalog and builds the dataframes used by every analysis notebook. It is executed by the other notebooks with `%run ./setup_data` and displays nothing on purpose (lighter published notebooks). The exploration of the raw data and the justification of each cleaning step are in `00_data_exploration`.

| Dataframe | Grain | Content |
|---|---|---|
| `games_df` | 1 row per game | cleaned columns (prices, dates, owners, reviews, ages, platforms) |
| `genres_df` | 1 row per (game, genre) | `genre` exploded |
| `languages_df` | 1 row per (game, language) | `language` exploded |
| `platforms_df` | 1 row per (game, platform) | Windows / Mac / Linux |
| `tags_df` | 1 row per (game, tag) | community tags and their vote count |

### Cleaning and feature engineering
- **prices** are strings in cents: converted to USD (`try_cast` returns null instead of failing on odd values, which matters on Spark 4 / serverless where ANSI mode is on).
- **release_date** comes as `yyyy/M/d` or only `yyyy/M`: parsed to a date, `release_year` and `release_month` extracted.
- **owners** is a range (`"20,000 .. 50,000"`): lower bound, upper bound and midpoint kept. It is an estimate of sales volume (SteamSpy).
- **reviews**: total, positive ratio and the Wilson lower bound of the positive ratio (ranks games by rating while penalising small review counts).
- **required_age**: first number extracted (`"MA 15+"` becomes 15).
- **estimated_revenue_usd** = owners midpoint x current price: a rough order of magnitude, not actual sales (prices change, discounts, regional pricing).
- Only `type == "game"` is kept; duplicated `appid` are removed.

## 1. Most represented genres
*Chart: bar, X = genre, Y = games.*

In [0]:
n_games = games_df.count()
genre_counts_df = (genres_df.groupBy("genre").agg(F.countDistinct("appid").alias("games"))
    .withColumn("pct_of_games", F.round(F.col("games") / n_games * 100, 1)).orderBy(F.desc("games")))
display(genre_counts_df)

genre,games,pct_of_games
Indie,39681,71.3
Action,23759,42.7
Casual,22086,39.7
Adventure,21431,38.5
Strategy,10895,19.6
Simulation,10836,19.5
RPG,9534,17.1
Early Access,6145,11.0
Free to Play,3393,6.1
Sports,2666,4.8


Databricks visualization. Run in Databricks to view.

## 2. Genres with a better positive / negative review ratio
Ratio = sum of positive reviews / sum of all reviews of the genre (weighted by reviews), for genres with at least 200 games.
*Chart: bar, X = genre, Y = positive_pct.*

In [0]:
genre_reviews_df = (genres_df.groupBy("genre")
    .agg(F.countDistinct("appid").alias("games"), F.sum("positive").alias("positive"), F.sum("negative").alias("negative"),
         F.round(F.percentile_approx("positive_ratio", 0.5) * 100, 1).alias("median_game_positive_pct"))
    .withColumn("positive_pct", F.round(F.col("positive") / (F.col("positive") + F.col("negative")) * 100, 1))
    .withColumn("positive_negative_ratio", F.round(F.col("positive") / F.col("negative"), 2))
    .filter(F.col("games") >= 200).orderBy(F.desc("positive_pct")))
display(genre_reviews_df)

genre,games,positive,negative,median_game_positive_pct,positive_pct,positive_negative_ratio
Animation & Modeling,322,690765,26392,75.0,96.3,26.17
Design & Illustration,406,674057,27007,76.5,96.1,24.96
Utilities,682,739335,43503,73.7,94.4,17.0
Indie,39681,32531023,4241234,80.0,88.5,7.67
Video Production,247,111514,16362,70.6,87.2,6.82
Casual,22086,10034967,1537296,80.3,86.7,6.53
Simulation,10836,15572390,2400512,74.9,86.6,6.49
Racing,2155,2340353,383691,75.4,85.9,6.1
RPG,9534,19425528,3274328,77.9,85.6,5.93
Action,23759,54858618,9687659,78.9,85.0,5.66


Databricks visualization. Run in Databricks to view.

## 3. Do publishers have favourite genres?
For the 15 publishers with the most games: share of their games in each genre, and their main genre.

In [0]:
top15 = [r["publisher"] for r in games_df.filter(F.col("publisher").isNotNull()).groupBy("publisher").count().orderBy(F.desc("count")).limit(15).collect()]
pub_genre_df = (genres_df.filter(F.col("publisher").isin(top15))
    .groupBy("publisher", "genre").agg(F.countDistinct("appid").alias("games"))
    .join(games_df.filter(F.col("publisher").isin(top15)).groupBy("publisher").agg(F.count("*").alias("publisher_games")), "publisher")
    .withColumn("pct_of_publisher_games", F.round(F.col("games") / F.col("publisher_games") * 100, 1)))
w = Window.partitionBy("publisher").orderBy(F.desc("games"))
favourite_genre_df = (pub_genre_df.withColumn("rank", F.row_number().over(w)).filter("rank = 1")
    .select("publisher", "publisher_games", F.col("genre").alias("favourite_genre"), "pct_of_publisher_games")
    .orderBy(F.desc("publisher_games")))
display(favourite_genre_df)

publisher,publisher_games,favourite_genre,pct_of_publisher_games
Big Fish Games,423,Casual,99.1
8floor,202,Casual,100.0
SEGA,165,Action,48.5
Strategy First,151,Strategy,34.4
Square Enix,141,Action,53.2
Choice of Games,140,RPG,99.3
HH-Games,132,Casual,100.0
Sekai Project,132,Casual,75.0
Ubisoft,127,Action,55.1
Laush Studio,126,Indie,98.4


Full matrix publisher x genre (share of the publisher's games). *Table: publisher, genre, pct_of_publisher_games.*

In [0]:
display(pub_genre_df.select("publisher", "genre", "pct_of_publisher_games").orderBy("publisher", F.desc("pct_of_publisher_games")))

publisher,genre,pct_of_publisher_games
8floor,Casual,100.0
8floor,Strategy,10.9
8floor,Simulation,5.0
8floor,Adventure,4.0
8floor,Indie,0.5
Alawar Entertainment,Casual,98.1
Alawar Entertainment,Adventure,88.8
Alawar Entertainment,Indie,71.0
Alawar Entertainment,Strategy,36.4
Alawar Entertainment,Simulation,3.7


In [0]:
# Ubisoft focus: what does our own catalogue on Steam look like?
display(pub_genre_df.filter(F.col("publisher") == "Ubisoft").orderBy(F.desc("games")))

publisher,genre,games,publisher_games,pct_of_publisher_games
Ubisoft,Action,70,127,55.1
Ubisoft,Adventure,45,127,35.4
Ubisoft,Strategy,22,127,17.3
Ubisoft,RPG,19,127,15.0
Ubisoft,Simulation,18,127,14.2
Ubisoft,Racing,14,127,11.0
Ubisoft,Casual,11,127,8.7
Ubisoft,Indie,7,127,5.5
Ubisoft,Free to Play,6,127,4.7
Ubisoft,Sports,5,127,3.9


## 4. Most lucrative genres
No sales figures in the dataset: estimated revenue = owners midpoint x current price (orders of magnitude only). Two views: **total** market of the genre and **mean per game** (pulled up by a few hits: the median is in the table). Free-to-play games earn through in-game purchases, invisible here.
*Charts: bar, X = genre, Y = total_estimated_revenue_musd; bar, X = genre, Y = avg_revenue_per_game_kusd.*

In [0]:
genre_revenue_df = (genres_df.filter(F.col("estimated_revenue_usd").isNotNull())
    .groupBy("genre")
    .agg(F.countDistinct("appid").alias("games"),
         F.round(F.sum("estimated_revenue_usd") / 1e6, 1).alias("total_estimated_revenue_musd"),
         F.round(F.percentile_approx("estimated_revenue_usd", 0.5) / 1e3, 1).alias("median_revenue_per_game_kusd"),
         F.round(F.avg("estimated_revenue_usd") / 1e3, 1).alias("avg_revenue_per_game_kusd"),
         F.round(F.percentile_approx("owners_mid", 0.5), 0).alias("median_owners"))
    .filter(F.col("games") >= 200).orderBy(F.desc("total_estimated_revenue_musd")))
display(genre_revenue_df)

genre,games,total_estimated_revenue_musd,median_revenue_per_game_kusd,avg_revenue_per_game_kusd,median_owners
Action,23759,58756.5,59.9,2473.0,10000.0
Adventure,21431,37245.7,69.9,1737.9,10000.0
Indie,39681,32346.6,49.9,815.2,10000.0
RPG,9534,27173.1,99.9,2850.1,10000.0
Strategy,10895,20150.0,79.9,1849.5,10000.0
Simulation,10836,18769.7,79.9,1732.2,10000.0
Casual,22086,8081.0,49.9,365.9,10000.0
Massively Multiplayer,1460,5930.2,0.0,4061.8,35000.0
Early Access,6145,5458.7,79.9,888.3,10000.0
Sports,2666,3149.9,69.7,1181.5,10000.0


Databricks visualization. Run in Databricks to view.

In [0]:
display(genre_revenue_df.orderBy(F.desc("avg_revenue_per_game_kusd")))

genre,games,total_estimated_revenue_musd,median_revenue_per_game_kusd,avg_revenue_per_game_kusd,median_owners
Massively Multiplayer,1460,5930.2,0.0,4061.8,35000.0
RPG,9534,27173.1,99.9,2850.1,10000.0
Action,23759,58756.5,59.9,2473.0,10000.0
Strategy,10895,20150.0,79.9,1849.5,10000.0
Adventure,21431,37245.7,69.9,1737.9,10000.0
Simulation,10836,18769.7,79.9,1732.2,10000.0
Racing,2155,2723.5,59.9,1263.8,10000.0
Sports,2666,3149.9,69.7,1181.5,10000.0
Early Access,6145,5458.7,79.9,888.3,10000.0
Animation & Modeling,322,265.2,49.9,823.6,10000.0


Databricks visualization. Run in Databricks to view.

## 5. Bonus: most used community tags
*Chart: bar, X = tag, Y = games.*

In [0]:
top_tags_df = (tags_df.groupBy("tag").agg(F.countDistinct("appid").alias("games"), F.sum("votes").alias("votes"))
    .orderBy(F.desc("games")).limit(25))
display(top_tags_df)

tag,games,votes
Indie,36062,2258563
Singleplayer,27344,2169116
Action,25022,2911335
Casual,23993,2007040
Adventure,23150,2412383
2D,14547,1468241
Strategy,11849,1398293
Simulation,11599,1278811
RPG,10161,1322315
Puzzle,9719,941809


Databricks visualization. Run in Databricks to view.

## Key figures (computed from the data)

In [0]:
gc = genre_counts_df.collect()
best = genre_reviews_df.first(); worst = genre_reviews_df.orderBy("positive_pct").first()
rev = genre_revenue_df.first(); per_game = genre_revenue_df.orderBy(F.desc("avg_revenue_per_game_kusd")).first()
print(f"- Most represented genres: {gc[0]['genre']} ({gc[0]['pct_of_games']}% of games), {gc[1]['genre']} ({gc[1]['pct_of_games']}%), {gc[2]['genre']} ({gc[2]['pct_of_games']}%).")
print(f"- Best review ratio: {best['genre']} ({best['positive_pct']}% positive); lowest: {worst['genre']} ({worst['positive_pct']}%).")
print(f"- Largest estimated market: {rev['genre']} ({rev['total_estimated_revenue_musd']:,} M USD); highest average revenue per game: {per_game['genre']} ({per_game['avg_revenue_per_game_kusd']:,} k USD).")

- Most represented genres: Indie (71.3% of games), Action (42.7%), Casual (39.7%).
- Best review ratio: Animation & Modeling (96.3% positive); lowest: Massively Multiplayer (73.1%).
- Largest estimated market: Action (58,756.5 M USD); highest average revenue per game: Massively Multiplayer (4,061.8 k USD).
